# 🤖 Qwen Personal Backend — PoC-003
Qwen3-8B · WebSocket to CF Durable Object · No tunnels · Auto idle-stop

In [ ]:
import subprocess, sys, os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

def install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

PACKAGES = [
    "transformers",
    "accelerate",
    "bitsandbytes",
    "ipywidgets",
    "websockets",
    "nest_asyncio",
]
for pkg in PACKAGES:
    install(pkg)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

vram_gb = torch.cuda.memory_allocated(0) / 1e9
print(f"✅ Model ready | VRAM: {vram_gb:.1f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import asyncio
import json
import time
from datetime import datetime
import nest_asyncio
import ipywidgets as widgets
import websockets
from IPython.display import display

nest_asyncio.apply()

# ── Inference ────────────────────────────────────────────────────────────────

def generate(messages: list, max_tokens: int = 800, temperature: float = 0.3) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ── UI widgets ───────────────────────────────────────────────────────────────

url_input = widgets.Text(
    value="",
    placeholder="https://your-worker.workers.dev",
    description="Worker URL:",
    layout=widgets.Layout(width="500px"),
)
idle_slider = widgets.IntSlider(
    value=30, min=5, max=120, step=5,
    description="Idle stop:",
    layout=widgets.Layout(width="350px"),
)
connect_btn = widgets.Button(
    description="▶ Connect",
    button_style="success",
    layout=widgets.Layout(width="140px"),
)
status_badge = widgets.HTML(
    value='<span style="background:#888;color:white;padding:3px 10px;border-radius:12px;font-size:13px">⬤ Disconnected</span>'
)
status_label = widgets.Label(value="Enter Worker URL and click Connect")
tasks_done   = widgets.Label(value="Tasks: 0")
log_output   = widgets.Textarea(
    value="",
    description="Log:",
    layout=widgets.Layout(width="600px", height="150px"),
    disabled=True,
)

task_counter = [0]

def set_status(connected: bool, text: str):
    if connected:
        status_badge.value = '<span style="background:#22c55e;color:white;padding:3px 10px;border-radius:12px;font-size:13px">⬤ Connected</span>'
    else:
        status_badge.value = '<span style="background:#ef4444;color:white;padding:3px 10px;border-radius:12px;font-size:13px">⬤ Disconnected</span>'
    status_label.value = text

def log(msg: str):
    ts  = datetime.now().strftime("%H:%M:%S")
    new = f"[{ts}] {msg}\n"
    log_output.value = new + log_output.value

# ── WebSocket loop ────────────────────────────────────────────────────────────

async def run_ws_loop(worker_url: str, idle_timeout_min: int = 30):
    ws_url = worker_url.replace("https://", "wss://").replace("http://", "ws://")
    ws_url = ws_url.rstrip("/") + "/connect"

    set_status(False, f"⏳ Connecting to {worker_url}...")
    log(f"Connecting to {ws_url}")

    # last_task_time tracks when we last finished a task (not when we received it)
    # This means idle timeout only counts actual idle time, not inference time
    last_task_time = time.time()

    try:
        async with websockets.connect(ws_url, ping_interval=30, ping_timeout=10) as ws:
            set_status(True, f"🔗 Connected · idle stop: {idle_timeout_min} min")
            log("WebSocket connected — ready for tasks")

            while True:
                idle_sec = time.time() - last_task_time
                if idle_sec > idle_timeout_min * 60:
                    set_status(False, f"⏹ Auto-stopped after {idle_timeout_min} min idle")
                    log(f"Auto-stopped: no tasks for {idle_timeout_min} min")
                    break

                try:
                    raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                except asyncio.TimeoutError:
                    idle_min = int(idle_sec // 60)
                    idle_s   = int(idle_sec % 60)
                    status_label.value = f"🔗 Connected · idle: {idle_min}m {idle_s}s / {idle_timeout_min}m"
                    continue
                except websockets.exceptions.ConnectionClosed:
                    set_status(False, "🔌 Connection closed by Worker")
                    log("Connection closed by Worker")
                    break

                try:
                    task = json.loads(raw)
                except Exception:
                    continue

                task_id     = task.get("id", "")
                messages    = task.get("messages", [])
                max_tokens  = task.get("max_tokens", 800)
                temperature = task.get("temperature", 0.3)

                set_status(True, f"⚙️ Generating... (task {task_id[:8]})")
                log(f"→ Task {task_id[:8]} | max_tokens={max_tokens}")

                try:
                    t0      = time.time()
                    content = generate(messages, max_tokens, temperature)
                    elapsed = time.time() - t0

                    # Reset idle timer AFTER task is fully done
                    last_task_time = time.time()

                    task_counter[0] += 1
                    tasks_done.value  = f"Tasks: {task_counter[0]}"
                    await ws.send(json.dumps({"id": task_id, "content": content}))
                    log(f"← Done | chars={len(content)} time={elapsed:.1f}s")
                    set_status(True, f"✅ Task done in {elapsed:.1f}s · idle reset")
                except Exception as e:
                    # Reset idle timer even on error — task was attempted
                    last_task_time = time.time()
                    log(f"← Error: {e}")
                    set_status(True, f"❌ Inference error: {e}")
                    await ws.send(json.dumps({"id": task_id, "error": str(e)}))

    except Exception as e:
        set_status(False, f"❌ Connection error: {e}")
        log(f"Connection error: {e}")

# ── Connect button ────────────────────────────────────────────────────────────

def on_connect(btn):
    worker_url = url_input.value.strip().rstrip("/")
    if not worker_url.startswith("https://"):
        set_status(False, "❌ URL must start with https://")
        return
    connect_btn.disabled = True
    try:
        asyncio.get_event_loop().run_until_complete(
            run_ws_loop(worker_url, idle_timeout_min=idle_slider.value)
        )
    finally:
        connect_btn.disabled = False

connect_btn.on_click(on_connect)

# ── Layout ────────────────────────────────────────────────────────────────────

display(widgets.VBox([
    widgets.Label("🔗 CF Worker URL:"),
    url_input,
    widgets.HBox([idle_slider, widgets.Label("min")]),
    widgets.HBox([connect_btn, status_badge, tasks_done]),
    status_label,
    log_output,
]))